In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Conf To Disable AQE

In [0]:
# spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

# spark.conf.set("spark.sql.adaptive.enabled", "false")  # AQE Disable

# 🚀 Spark Jobs, Stages & Tasks (Interview-Level Deep Dive)

---

# 1️⃣ What is a Job in Spark?

## 📌 Definition

> A Job is triggered when an **action** is called on an RDD or DataFrame.

---

## 🔹 Examples of Actions

```python
df.show()
df.collect()
df.count()
df.write.parquet("/path")
```

Each action → **New Job**

---

## 🔹 Example

```python
df = spark.read.csv("file.csv")

df_filtered = df.filter("age > 25")

df_filtered.show()   # Job 1
df_filtered.count()  # Job 2
```

👉 Two actions → **Two separate jobs**

---

## 🔹 Key Points

- One action = One job  
- Job is the **top-level execution unit**  
- Visible in **Spark UI → Jobs tab**

---

# 2️⃣ What is a Stage?

## 📌 Definition

> A Stage is a group of tasks that can be executed **without shuffle**.

---

## 🔹 How Stages Are Created?

- Spark divides a job into stages
- Each **wide transformation (shuffle)** creates a new stage

---

## 🔹 Example

```python
rdd = spark.sparkContext.parallelize([1,2,3,4])

rdd2 = rdd.map(lambda x: x * 2)
rdd3 = rdd2.filter(lambda x: x > 2)
rdd4 = rdd3.reduceByKey(lambda x,y: x+y)

rdd4.collect()
```

---

## 🔹 Stage Breakdown

```
Stage 1: map → filter   (Narrow transformations)
Stage 2: reduceByKey    (Wide transformation → shuffle)
```

---

## 🔹 Key Points

- Narrow transformations → Same stage  
- Wide transformations → New stage  
- Shuffle = Stage boundary  

---

# 3️⃣ What is a Task?

## 📌 Definition

> A Task is the smallest unit of execution in Spark.

---

## 🔹 Key Rule

👉 One partition = One task  

---

## 🔹 Example

```python
rdd = spark.sparkContext.parallelize(range(1, 101), 4)

rdd.map(lambda x: x * 2).collect()
```

- Total partitions = 4  
- Total tasks = 4  

Each task processes one partition.

---

## 🔹 Where Tasks Run?

- Inside **Executors**
- In parallel across cluster

---

# 4️⃣ Complete Flow: Job → Stage → Task

---

## 🔹 Execution Flow

```
Action Triggered
      ↓
Job Created
      ↓
DAG Scheduler splits into stages
      ↓
Task Scheduler creates tasks
      ↓
Tasks assigned to Executors
      ↓
Execution happens
```

---

## 🔹 Visual Representation

```
Job
 ├── Stage 1 (No Shuffle)
 │     ├── Task 1
 │     ├── Task 2
 │     └── Task 3
 │
 └── Stage 2 (Shuffle)
       ├── Task 1
       ├── Task 2
       └── Task 3
```

---

# 5️⃣ Hands-On Example (Very Important)

```python
df = spark.range(0, 100)

df2 = df.filter("id > 10")      # Narrow
df3 = df2.groupBy("id").count() # Wide

df3.show()
```

---

## 🔹 What Happens Internally?

### Step 1: Action Triggered

```
show() → Job created
```

---

### Step 2: DAG Created

```
range → filter → groupBy → count
```

---

### Step 3: Stage Division

```
Stage 1: range + filter
Stage 2: groupBy (shuffle)
```

---

### Step 4: Tasks Created

If 4 partitions:

```
Stage 1 → 4 tasks
Stage 2 → 4 tasks
```

---

# 6️⃣ How to See This in Spark UI

---

## 🔹 Steps

1. Go to Databricks notebook  
2. Run any action (`show()`, `display()`)  
3. Click → "View Spark UI"  

---

## 🔹 What You Will See

### Jobs Tab

- List of jobs triggered  
- Execution time  

---

### Stages Tab

- Stage breakdown  
- Shuffle read/write  
- Task count  

---

### Tasks View

- Task duration  
- Input size  
- Skew detection  

---

# 7️⃣ Important Interview Concepts

---

## 🔹 Job vs Stage vs Task

| Level | Description |
|--------|-------------|
| Job | Triggered by action |
| Stage | Split by shuffle |
| Task | Runs on partition |

---

## 🔹 Key Relationships

- One Job → Multiple Stages  
- One Stage → Multiple Tasks  
- One Task → One Partition  

---

# 8️⃣ Advanced Concepts

---

## 🔹 Stage Types

- **Shuffle Map Stage** → Produces shuffle data  
- **Result Stage** → Final stage returning output  

---

## 🔹 Example

```
Stage 1 → Shuffle Map Stage
Stage 2 → Result Stage
```

---

## 🔹 DAG Scheduler vs Task Scheduler

### DAG Scheduler

- Splits job into stages  
- Handles shuffle dependencies  

---

### Task Scheduler

- Assigns tasks to executors  
- Handles execution  

---

# 9️⃣ Real-World Insight

---

## 🔹 Performance Impact

- More partitions → More tasks → Better parallelism  
- Too many tasks → Overhead  
- Shuffle → Expensive  

---

## 🔹 Optimization Tips

- Reduce shuffle where possible  
- Use broadcast joins  
- Tune partitions  
- Monitor Spark UI  

---

# 🎯 Interview-Level Summary

- Action → Creates Job  
- Job → Split into Stages  
- Stage → Split into Tasks  
- Task → Executes on partition  
- Shuffle → Creates new stage  
- Narrow → Same stage  
- Wide → New stage  

---

# 🚀 Final Understanding

```
Action
  ↓
Job
  ↓
Stages (Based on Shuffle)
  ↓
Tasks (Based on Partitions)
  ↓
Executors execute tasks
```

---

# 🔥 Golden Rule (Must Remember)

👉 One Partition = One Task  
👉 One Action = One Job  
👉 One Shuffle = New Stage  

In [0]:
dbutils.fs.ls("dbfs:/FileStore/tables/Arijit/Test/")

In [0]:
df = spark.read.format("csv").option("header", True)\
                            .option("inferSchema", True)\
                            .load("dbfs:/FileStore/tables/Arijit/Test/MegaMart.csv")

In [0]:
df.display()

In [0]:
# Transformations

df = df.filter(col('product_name') == 'Sneakers')

In [0]:
df = df.select('order_id', 'product_name')

In [0]:
df = df.groupBy('product_name').agg(count(col('order_id')))

In [0]:
display(df)

# 🚀 Spark Joins Deep Dive (Interview-Level)

---

# 1️⃣ What is a Join in Spark?

## 📌 Definition

> A join combines two datasets based on a common key.

---

## 🔹 Example

```python
df1.join(df2, "id")
```

---

# 2️⃣ Types of Joins in Spark

---

## 🔹 1. Inner Join

Returns matching records from both tables.

```python
df1.join(df2, "id", "inner")
```

---

## 🔹 2. Left Join

All records from left + matching from right.

```python
df1.join(df2, "id", "left")
```

---

## 🔹 3. Right Join

All records from right + matching from left.

```python
df1.join(df2, "id", "right")
```

---

## 🔹 4. Full Outer Join

All records from both sides.

```python
df1.join(df2, "id", "outer")
```

---

## 🔹 5. Left Semi Join

Only matching rows from left (no columns from right).

```python
df1.join(df2, "id", "left_semi")
```

---

## 🔹 6. Left Anti Join

Rows from left that DO NOT match.

```python
df1.join(df2, "id", "left_anti")
```

---

# 3️⃣ How Join Works Internally

---

## 🔥 Step-by-Step Execution

```
1. Read both datasets
2. Partition data based on join key
3. Shuffle data across executors
4. Bring same keys together
5. Perform join
```

---

## 🔹 Why Shuffle Happens?

Because:

- Same keys must be in same partition
- Data initially distributed randomly

---

## 🔹 Example

```
df1:
Partition 1 → id: 1,2
Partition 2 → id: 3,4

df2:
Partition 1 → id: 3,4
Partition 2 → id: 1,2
```

Before join → keys are scattered

After shuffle:

```
Partition 1 → id: 1,2
Partition 2 → id: 3,4
```

Now join is possible.

---

# 4️⃣ Hash Partitioning (Very Important 🔥)

---

## 📌 What is Hash Partitioning?

Spark distributes data using:

```
partition = hash(key) % num_partitions
```

---

## 🔹 Example

```python
hash("A") % 4 = 1
hash("B") % 4 = 3
```

So:

- All "A" → Partition 1
- All "B" → Partition 3

---

## 🔹 Why Important?

- Ensures same keys go to same partition
- Enables join & aggregation
- Used during shuffle

---

# 5️⃣ Join Strategies (Most Important 🔥🔥🔥)

Spark uses different strategies based on data size.

---

# 🔹 1. Broadcast Hash Join (Best for small table)

---

## 📌 When Used?

- One table is small (< 10MB default)

---

## 🔹 Example

```python
from pyspark.sql.functions import broadcast

df1.join(broadcast(df2), "id")
```

---

## 🔹 How It Works?

```
Small table → Sent to all executors
Large table → Stays distributed
Join happens locally
```

---

## 🔹 Advantages

- No shuffle for large table
- Very fast

---

# 🔹 2. Sort Merge Join (Default for large data)

---

## 📌 When Used?

- Both tables are large

---

## 🔹 Steps

```
1. Shuffle both datasets
2. Sort data by key
3. Merge join
```

---

## 🔹 Characteristics

- Requires shuffle
- Requires sorting
- More expensive

---

# 🔹 3. Shuffle Hash Join

---

## 📌 When Used?

- Smaller data compared to SMJ
- Enough memory available

---

## 🔹 Steps

```
1. Shuffle both datasets
2. Build hash table on smaller partition
3. Probe with larger dataset
```

---

# 🔹 4. Broadcast Nested Loop Join

---

## 📌 When Used?

- No join key (cross join)
- Non-equi joins

---

## 🔹 Very expensive (avoid)

---

# 6️⃣ How to Check Join Type

---

```python
df.join(df2, "id").explain(True)
```

Look for:

- BroadcastHashJoin
- SortMergeJoin
- ShuffledHashJoin

---

# 7️⃣ Shuffle in Join (Deep Concept)

---

## 🔹 Why Shuffle Happens?

Because:

- Data is not aligned by key
- Needs redistribution

---

## 🔹 Shuffle Cost

- Disk I/O
- Network transfer
- Serialization
- Memory usage

---

## 🔹 Spark Config

```python
spark.conf.get("spark.sql.shuffle.partitions")
```

Default = 200

---

# 8️⃣ Data Skew in Join (Very Important)

---

## 🔹 What is Skew?

Uneven key distribution.

Example:

```
Key "A" → 90% data
Key "B" → 10% data
```

---

## 🔹 Problem

- One partition overloaded
- One task slow
- Others idle

---

## 🔹 Solutions

- Broadcast join
- Salting technique
- Repartition
- Skew join optimization (Spark 3+)

---

# 9️⃣ Partitioning Optimization

---

## 🔹 Pre-Partition Data

```python
df1 = df1.repartition("id")
df2 = df2.repartition("id")
```

Helps reduce shuffle.

---

## 🔹 Bucketing (Advanced)

- Pre-partition data on disk
- Used in Hive tables

---

# 🔟 Hands-On Example

---

```python
df1 = spark.range(0, 1000000)
df2 = spark.range(0, 1000)

# Normal join (shuffle)
df1.join(df2, "id").explain()

# Broadcast join (optimized)
from pyspark.sql.functions import broadcast
df1.join(broadcast(df2), "id").explain()
```

---

# 🎯 Interview-Level Key Points

- Join is usually a **wide transformation**
- Shuffle is required to align keys
- Hash partitioning ensures correct grouping
- Broadcast join avoids shuffle
- Sort merge join is default for large data
- Skew is biggest performance issue
- Always check execution plan

---

# 🚀 Final Summary

```
Join
  ↓
Shuffle (if needed)
  ↓
Hash partitioning
  ↓
Join strategy selected
  ↓
Execution
```

---

# 🔥 Golden Rules

- Small table → Broadcast join  
- Large tables → Sort merge join  
- Avoid shuffle when possible  
- Watch skew carefully  
- Always check df.explain()

# 🚀 Advanced Spark Join Optimization (Deep Dive)

---

# 1️⃣ Broadcast Join vs Sort Merge Join (Real Understanding)

---

## 🔹 Broadcast Hash Join (BHJ)

### 📌 When Used?

- One dataset is small (default < 10MB)
- Can fit in memory

---

### 🔹 Example

```python
from pyspark.sql.functions import broadcast

df_large = spark.range(0, 1000000)
df_small = spark.range(0, 1000)

df_large.join(broadcast(df_small), "id").explain(True)
```

---

### 🔹 How It Works

```
Small table → Broadcast to all executors
Large table → Remains distributed
Join happens locally
```

---

### 🔹 Benefits

- No shuffle for large dataset
- Very fast
- Low network cost

---

## 🔹 Sort Merge Join (SMJ)

### 📌 When Used?

- Both datasets are large
- Default join strategy

---

### 🔹 Example

```python
df1 = spark.range(0, 1000000)
df2 = spark.range(0, 1000000)

df1.join(df2, "id").explain(True)
```

---

### 🔹 How It Works

```
1. Shuffle both datasets
2. Sort each partition
3. Merge join
```

---

### 🔹 Drawback

- Expensive (shuffle + sort)
- High disk + network I/O

---

## 🔥 Key Comparison

| Feature | Broadcast Join | Sort Merge Join |
|----------|----------------|------------------|
| Shuffle | ❌ No (big table) | ✅ Yes |
| Speed | Fast | Slower |
| Use Case | Small + Large | Large + Large |

---

# 2️⃣ Salting Technique (Handle Data Skew)

---

## 🔹 Problem: Data Skew

Example:

```
Key "A" → 90% data
Key "B" → 10% data
```

One partition overloaded → slow job.

---

## 🔹 Solution: Salting

Add random suffix to skewed keys.

---

## 🔹 Step-by-Step Example

### Step 1: Add Salt Column

```python
from pyspark.sql.functions import rand, floor

df1_salted = df1.withColumn("salt", floor(rand() * 5))
```

---

### Step 2: Expand Small Dataset

```python
from pyspark.sql.functions import explode, array

df2_salted = df2.withColumn("salt", explode(array([0,1,2,3,4])))
```

---

### Step 3: Join Using Salt + Key

```python
df_joined = df1_salted.join(df2_salted, ["id", "salt"])
```

---

## 🔹 What Happens?

- Skewed key "A" split into multiple partitions
- Load distributed evenly

---

# 3️⃣ Skew Join Handling (Spark 3+)

---

## 🔹 Automatic Skew Handling

Spark 3 introduced:

```
Adaptive Query Execution (AQE)
```

---

## 🔹 Enable It

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

---

## 🔹 What Spark Does

- Detects skewed partitions
- Splits large partitions
- Joins them separately

---

## 🔹 Benefit

- No manual salting required
- Automatic optimization

---

# 4️⃣ Bucketing vs Partitioning

---

# 🔹 Partitioning

Data physically divided by column.

```python
df.write.partitionBy("country").parquet("/path")
```

---

## 🔹 Use Case

- Query pruning
- Faster filtering

---

## 🔹 Bucketing

Data divided into fixed number of buckets using hash.

```python
df.write.bucketBy(8, "id").saveAsTable("bucketed_table")
```

---

## 🔹 Key Difference

| Feature | Partitioning | Bucketing |
|----------|--------------|-------------|
| Based on | Column value | Hash |
| File structure | Folder-based | Fixed buckets |
| Use case | Filtering | Joins |

---

## 🔹 Why Bucketing Helps Joins?

If both tables:

- Bucketed on same key
- Same number of buckets

👉 Shuffle can be avoided.

---

# 5️⃣ Real-World Join Optimization Case Study

---

## 🔹 Problem

- Large table: 1 Billion rows
- Small table: 1 Million rows
- Join taking too long

---

## 🔹 Initial Code

```python
df_large.join(df_small, "id")
```

---

## 🔹 Issues

- Sort Merge Join
- Full shuffle
- High execution time

---

## 🔹 Optimized Solution

```python
from pyspark.sql.functions import broadcast

df_large.join(broadcast(df_small), "id")
```

---

## 🔹 Result

- Broadcast Hash Join used
- Shuffle avoided
- Execution time reduced drastically

---

## 🔹 Additional Optimization

- Reduce shuffle partitions:

```python
spark.conf.set("spark.sql.shuffle.partitions", "50")
```

---

## 🔹 If Skew Exists

Apply:

- Salting
- AQE skew handling

---

# 🎯 Interview-Level Key Points

- Broadcast join → Best for small table
- Sort merge join → Default for large data
- Skew → Major performance issue
- Salting → Manual skew handling
- AQE → Automatic skew handling
- Bucketing → Avoid shuffle in joins
- Partitioning → Helps filtering

---

# 🚀 Final Summary

```
Join Optimization Strategy:

Small table → Broadcast
Large tables → Sort Merge
Skew → Salting / AQE
Pre-partitioned → Less shuffle
Bucketed tables → No shuffle (ideal case)
```

---

# 🔥 Golden Rules

- Always check df.explain()
- Avoid unnecessary shuffle
- Use broadcast wisely
- Watch skew carefully
- Tune shuffle partitions

# 🚀 Spark Join Hands On

## Disabling Auto Broadcast Join & AQE For Understanding Normal Sort Merge Join

In [0]:
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.enabled", False)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType
# schema = StructType([
#     StructField('order_id', LongType(), True),
#     StructField('product_name', StringType(), True),
#     StructField('order_date', StringType(), True),
#     StructField('order_customer_id', LongType(), True),
#     StructField('order_status', StringType(), True)
# ])

# Create First DataFrame
data1 = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David"),
    (5, "Eve")
]

df1 = spark.createDataFrame(data1, ["id", "name"])

# Create Second DataFrame
data2 = [
    (1, 50000),
    (2, 60000),
    (3, 70000),
    (6, 80000),
    (8, 90000)
]

df2 = spark.createDataFrame(data2, ["id", "salary"])



## Sort Merge Join

![image_1775982330032.png](./Images/image_1775982330032.png "image_1775982330032.png")

In [0]:
# df_join = df1.join(df2, df1['id'] == df2['id'], how="left")
df_join = df1.join(df2, df1.id == df2.id, how="left")

In [0]:
df_join.display()

## Broadcast Hash Join

![image_1775982296625.png](./Images/image_1775982296625.png "image_1775982296625.png")

In [0]:
from pyspark.sql.functions import broadcast

df_join_broad = df1.join(broadcast(df2), df1.id == df2.id, how="left")

In [0]:
df_join_broad.display()

# 🚀 Spark Performance Mastery (Memory, Optimization, Debugging)

---

# 1️⃣ Spark Memory Tuning (Very Important 🔥)

---

## 🔹 Spark Memory Basics

Each Executor has:

```
Total Memory
   ├── Execution Memory (for shuffle, joins, aggregations)
   └── Storage Memory (for caching)
```

---

## 🔹 Key Configurations

```python
spark.conf.set("spark.executor.memory", "4g")
spark.conf.set("spark.executor.cores", "2")
spark.conf.set("spark.sql.shuffle.partitions", "200")
```

---

## 🔹 Important Memory Concepts

### 1. Execution Memory

Used for:

- Shuffle
- Join
- Aggregation

---

### 2. Storage Memory

Used for:

- Cache / persist
- Broadcast variables

---

### 3. Unified Memory Manager

Spark dynamically shares memory between:

- Execution
- Storage

---

## 🔹 Common Memory Issues

### ❌ OutOfMemoryError

Causes:

- Using `collect()`
- Large joins
- Skew
- Too many partitions

---

### ❌ GC Overhead

- Too many small objects
- High memory pressure

---

## 🔹 Tuning Tips

- Increase executor memory if needed  
- Avoid `collect()`  
- Use broadcast joins  
- Reduce shuffle partitions  
- Cache only when necessary  

---

# 2️⃣ End-to-End Spark Job Optimization Checklist

---

## 🔹 Step 1: Data Reading

- Use column pruning:

```python
df.select("id", "name")
```

- Use predicate pushdown:

```python
df.filter("age > 25")
```

---

## 🔹 Step 2: Partition Optimization

- Avoid too few partitions  
- Avoid too many partitions  

```python
df.repartition(50)
```

---

## 🔹 Step 3: Join Optimization

- Use broadcast for small tables  
- Avoid unnecessary joins  
- Check join strategy:

```python
df.explain()
```

---

## 🔹 Step 4: Reduce Shuffle

- Avoid groupBy when possible  
- Use reduceByKey instead of groupByKey (RDD)  
- Use partitioning wisely  

---

## 🔹 Step 5: Handle Skew

- Use salting  
- Enable AQE  

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
```

---

## 🔹 Step 6: Caching Strategy

```python
df.cache()
```

Use only when:

- Data reused multiple times  

---

## 🔹 Step 7: Writing Data

- Avoid too many output files  
- Use coalesce:

```python
df.coalesce(10).write.parquet("/path")
```

---

# 3️⃣ Real Interview Questions on Joins

---

## ❓ Q1: Which join is fastest?

👉 Broadcast Join (if one table is small)

---

## ❓ Q2: Why is join expensive?

👉 Because of shuffle and data movement

---

## ❓ Q3: How to avoid shuffle in join?

- Broadcast join  
- Bucketing  
- Pre-partitioned data  

---

## ❓ Q4: What is data skew in join?

👉 Uneven key distribution causing slow tasks

---

## ❓ Q5: How to handle skew?

- Salting  
- AQE  
- Repartition  

---

## ❓ Q6: Difference between Sort Merge and Broadcast Join?

| Feature | Broadcast | Sort Merge |
|----------|------------|-------------|
| Shuffle | ❌ | ✅ |
| Speed | Fast | Slower |
| Use case | Small table | Large tables |

---

## ❓ Q7: How to check join type?

```python
df.explain(True)
```

---

# 4️⃣ Debugging Slow Spark Jobs (Spark UI Deep Dive)

---

## 🔹 Step 1: Open Spark UI

- Databricks → "View Spark UI"

---

## 🔹 Step 2: Check Jobs Tab

- Identify slow job  
- Check duration  

---

## 🔹 Step 3: Check Stages Tab

Look for:

- Long-running stages  
- Shuffle read/write  

---

## 🔹 Step 4: Check Tasks

Inside stage:

- Task duration  
- Input size  
- Skew detection  

---

## 🔹 Signs of Problems

---

### 🔥 Data Skew

- One task very slow  
- Others finish quickly  

---

### 🔥 Too Many Tasks

- High overhead  
- Small data per task  

---

### 🔥 Shuffle Bottleneck

- High shuffle read/write  
- Disk spill  

---

### 🔥 Memory Issues

- Spill to disk  
- GC overhead  

---

# 5️⃣ Practical Debugging Example

---

## Scenario: Slow Join

```python
df1.join(df2, "id")
```

---

## 🔹 Debug Steps

1. Check execution plan:

```python
df.explain(True)
```

2. Check Spark UI:

- Shuffle size  
- Stage time  

---

## 🔹 Fix

- Use broadcast:

```python
df1.join(broadcast(df2), "id")
```

---

## 🔹 Result

- Shuffle reduced  
- Faster execution  

---

# 🎯 Interview-Level Key Points

- Memory tuning = Critical for performance  
- Shuffle = Most expensive operation  
- Broadcast = Best optimization  
- Spark UI = Debugging tool  
- Skew = Major bottleneck  
- Partition tuning = Key to parallelism  

---

# 🚀 Final Summary

```
Performance = Memory + Partitioning + Join Strategy + Shuffle Control
```

---

# 🔥 Golden Rules

- Avoid collect()  
- Use broadcast wisely  
- Monitor Spark UI  
- Tune partitions  
- Handle skew early  
- Cache only when needed  

# 🚀 50+ Advanced Spark & Data Engineering Interview Questions (DP-203 + Product Companies)

---

# 🔥 SECTION 1: Spark Core Concepts

---

## ❓1. What is Spark and why is it faster than Hadoop?

👉 Spark is an in-memory distributed processing engine.  
👉 Faster than Hadoop because it avoids disk I/O using memory + DAG execution.

---

## ❓2. What is Lazy Evaluation?

👉 Spark delays execution until an action is triggered, allowing optimization.

---

## ❓3. Difference between Transformation and Action?

- Transformation → Lazy (map, filter)
- Action → Triggers execution (count, collect)

---

## ❓4. What is DAG?

👉 Directed Acyclic Graph representing execution plan of transformations.

---

## ❓5. What is SparkSession?

👉 Unified entry point to Spark (replaces SparkContext, SQLContext).

---

# 🔥 SECTION 2: RDD vs DataFrame

---

## ❓6. RDD vs DataFrame?

| RDD | DataFrame |
|------|------------|
| No schema | Schema |
| No optimization | Catalyst optimized |
| Slower | Faster |

---

## ❓7. Why DataFrame is faster?

👉 Uses Catalyst optimizer + Tungsten execution engine.

---

## ❓8. What is Catalyst Optimizer?

👉 Query optimizer that improves execution plan using rules like predicate pushdown.

---

## ❓9. What is Tungsten?

👉 Execution engine that improves CPU and memory efficiency.

---

# 🔥 SECTION 3: Partitioning & Transformations

---

## ❓10. What is partition in Spark?

👉 Small chunk of distributed data processed by a task.

---

## ❓11. What is narrow transformation?

👉 No shuffle, one-to-one partition dependency.

---

## ❓12. What is wide transformation?

👉 Requires shuffle, many-to-many dependency.

---

## ❓13. What is shuffle?

👉 Data redistribution across executors based on key.

---

## ❓14. Why is shuffle expensive?

👉 Disk I/O + network transfer + serialization.

---

## ❓15. Repartition vs Coalesce?

| Repartition | Coalesce |
|--------------|-----------|
| Shuffle | No shuffle |
| Increase/decrease | Only decrease |

---

# 🔥 SECTION 4: Joins (VERY IMPORTANT)

---

## ❓16. Types of joins in Spark?

- Inner
- Left
- Right
- Full
- Left Semi
- Left Anti

---

## ❓17. What is broadcast join?

👉 Small table is sent to all executors to avoid shuffle.

---

## ❓18. When to use broadcast join?

👉 When one dataset is small (<10MB default).

---

## ❓19. What is sort merge join?

👉 Default join for large datasets (shuffle + sort + merge).

---

## ❓20. How to check join type?

```python
df.explain(True)
```

---

## ❓21. What is data skew?

👉 Uneven distribution of data across partitions.

---

## ❓22. How to handle skew?

- Salting
- AQE
- Repartition
- Broadcast join

---

# 🔥 SECTION 5: Spark Execution

---

## ❓23. What is a Job?

👉 Triggered by an action.

---

## ❓24. What is a Stage?

👉 Group of tasks without shuffle.

---

## ❓25. What is a Task?

👉 Smallest execution unit (per partition).

---

## ❓26. Relation?

👉 Job → Stage → Task

---

## ❓27. What creates stage boundary?

👉 Shuffle (wide transformation)

---

# 🔥 SECTION 6: Performance Optimization

---

## ❓28. Why collect() is dangerous?

👉 Brings all data to driver → OOM.

---

## ❓29. How to optimize Spark job?

- Reduce shuffle
- Use broadcast
- Tune partitions
- Use caching wisely

---

## ❓30. Ideal partition size?

👉 ~100–200 MB per partition

---

## ❓31. What is caching?

```python
df.cache()
```

👉 Stores data in memory for reuse.

---

## ❓32. When NOT to cache?

👉 When data is used only once.

---

# 🔥 SECTION 7: Spark UI & Debugging

---

## ❓33. How to debug slow job?

👉 Use Spark UI:

- Jobs tab
- Stages tab
- Tasks tab

---

## ❓34. What indicates skew?

👉 One task much slower than others.

---

## ❓35. What is spill?

👉 Data moved to disk due to memory shortage.

---

# 🔥 SECTION 8: Advanced Concepts

---

## ❓36. What is AQE?

👉 Adaptive Query Execution (runtime optimization).

---

## ❓37. What is bucketing?

👉 Pre-hashing data into fixed buckets to avoid shuffle.

---

## ❓38. Partitioning vs Bucketing?

| Partitioning | Bucketing |
|--------------|------------|
| Folder-based | Hash-based |
| Used for filtering | Used for joins |

---

## ❓39. What is lineage?

👉 DAG of transformations used for fault tolerance.

---

## ❓40. What is checkpointing?

👉 Saves RDD to storage to truncate lineage.

---

# 🔥 SECTION 9: Azure DP-203 Specific

---

## ❓41. Difference between ADF and Databricks?

- ADF → Orchestration
- Databricks → Processing

---

## ❓42. What is Delta Lake?

👉 Storage layer with ACID + versioning.

---

## ❓43. What is Medallion Architecture?

👉 Bronze → Silver → Gold

---

## ❓44. What is Event Hub?

👉 Streaming ingestion service.

---

## ❓45. How to handle duplicates in streaming?

👉 Windowing / watermarking / dedup logic.

---

# 🔥 SECTION 10: Real Product Company Questions

---

## ❓46. How do you optimize a slow join?

👉 Broadcast + reduce shuffle + handle skew.

---

## ❓47. How do you decide number of partitions?

👉 Based on data size and cluster cores.

---

## ❓48. What happens when executor fails?

👉 Spark recomputes using lineage.

---

## ❓49. Why is Spark fault-tolerant?

👉 Because of lineage (recomputation).

---

## ❓50. Difference between persist and cache?

👉 cache = MEMORY_ONLY  
👉 persist = custom storage level

---

## ❓51. How do you reduce small files problem?

👉 Use coalesce before write.

---

## ❓52. What is driver vs executor?

- Driver → Orchestrates
- Executor → Executes tasks

---

## ❓53. What is broadcast variable?

👉 Shared read-only variable across executors.

---

## ❓54. What is accumulator?

👉 Used for counters across tasks.

---

# 🎯 FINAL INTERVIEW STRATEGY

---

## 🔥 What Interviewers Focus On

- Join optimization
- Shuffle understanding
- Partition tuning
- Spark UI debugging
- Real-world problem solving

---

## 🚀 Golden Tip

👉 Always explain with:

- Example  
- Performance impact  
- Optimization approach  

---

# 🔥 FINAL SUMMARY

- Spark = Distributed + In-memory
- Shuffle = Most expensive
- Broadcast = Best optimization
- Skew = Biggest problem
- Spark UI = Debugging tool
- Partitioning = Key to performance

# 🚀 Driver vs Executor Memory Management in Spark (Deep Dive)

---

# 1️⃣ Overview

Spark uses a **distributed memory model**:

```
Driver (Control Plane)
        ↓
Executors (Data Processing Layer)
```

Both have separate memory responsibilities.

---

# 2️⃣ Driver Memory Management

---

## 📌 What is Driver?

Driver is the **brain of Spark application**.

It:

- Runs SparkSession
- Builds DAG
- Schedules tasks
- Collects results

---

## 🔹 Driver Memory Usage

Driver memory is used for:

- Task scheduling metadata
- DAG execution plan
- Collecting results (`collect()`, `toPandas()`)
- Broadcast variables metadata
- Accumulators

---

## 🔹 Driver Memory Configuration

```python
spark.conf.set("spark.driver.memory", "4g")
```

---

## 🔹 Driver Memory Flow

```
Executors → Send results → Driver → Store in memory
```

---

## 🔥 Common Driver Memory Issues

---

### ❌ 1. OutOfMemoryError (OOM)

Cause:

```python
df.collect()
```

👉 Entire dataset moves to driver

---

### ❌ 2. toPandas() Crash

```python
df.toPandas()
```

👉 Loads full data into driver (very dangerous)

---

### ❌ 3. Large Broadcast Variables

- Large object sent to driver first
- Then distributed

---

## 🔹 Best Practices

- Avoid `collect()` on large data  
- Avoid `toPandas()`  
- Use `limit()` instead  
- Keep driver lightweight  

---

![image_1776006003546.png](./Images/image_1776006003546.png "image_1776006003546.png")<br><br>
![image_1776006052448.png](./Images/image_1776006052448.png "image_1776006052448.png")<br><br>

# 3️⃣ Executor Memory Management

---

## 📌 What is Executor?

Executors are worker processes.

They:

- Execute tasks
- Store partitions
- Perform shuffle

---

## 🔹 Executor Memory Structure

```
Executor Memory
   ├── Execution Memory
   └── Storage Memory
```

---

## 🔹 Configuration

```python
spark.conf.set("spark.executor.memory", "8g")
spark.conf.set("spark.executor.cores", "4")
```

---

# 4️⃣ Unified Memory Management (Very Important 🔥)

---

## 🔹 Spark Memory Model

```
Total Executor Memory
        ↓
Unified Memory Pool
   ├── Execution Memory
   └── Storage Memory
```

---

## 🔹 Behavior

- Execution can borrow from Storage  
- Storage can borrow from Execution  
- Dynamic allocation  

---

## 🔹 Default Split

~60% for execution + storage  
~40% reserved (user memory, overhead)

---

# 5️⃣ Execution Memory

---

## 🔹 Used For

- Shuffle operations
- Join operations
- Aggregations
- Sort operations

---

## 🔹 Example

```python
df.groupBy("id").count()
```

👉 Uses execution memory

---

## 🔹 Problem

If insufficient:

👉 Spill to disk (slow)

---

# 6️⃣ Storage Memory

---

## 🔹 Used For

- Cache / persist
- Broadcast variables

---

## 🔹 Example

```python
df.cache()
```

---

## 🔹 Problem

If insufficient:

👉 Cached data evicted

---

# 7️⃣ Memory Spill (Very Important)

---

## 🔹 What is Spill?

When memory is full:

```
Memory → Disk (spill)
```

---

## 🔹 Causes

- Large shuffle
- Large joins
- Insufficient memory

---

## 🔹 Impact

- Slower performance
- Disk I/O increase

---

# 8️⃣ Driver vs Executor Comparison

---

| Feature | Driver | Executor |
|----------|----------|-------------|
| Role | Orchestrator | Worker |
| Stores data? | Limited | Yes |
| Runs tasks? | ❌ | ✅ |
| Memory usage | Metadata + results | Data processing |
| OOM risk | collect(), toPandas() | shuffle, joins |

---

# 9️⃣ Real-World Example

---

## Scenario: Large Join

```python
df1.join(df2, "id")
```

---

## 🔹 What Happens?

- Executors perform join  
- Shuffle happens  
- Execution memory used  
- If memory low → spill  

---

## 🔹 If You Do:

```python
df.collect()
```

👉 Entire result moves to driver → crash

---

# 🔟 Advanced Configurations

---

## 🔹 Shuffle Partitions

```python
spark.conf.set("spark.sql.shuffle.partitions", "100")
```

---

## 🔹 Memory Fraction (Advanced)

```python
spark.memory.fraction
spark.memory.storageFraction
```

---

## 🔹 Executor Instances

```python
spark.executor.instances
```

---

# 1️⃣1️⃣ Performance Tuning Tips

---

## 🔥 For Driver

- Avoid large data collection  
- Keep minimal logic  
- Use sampling  

---

## 🔥 For Executors

- Increase executor memory  
- Tune partitions  
- Reduce shuffle  
- Use broadcast joins  

---

## 🔥 General

- Monitor Spark UI  
- Watch spill metrics  
- Optimize joins  

---

# 🎯 Interview-Level Questions

---

## ❓ Why does driver crash?

👉 Because large data is collected into driver memory.

---

## ❓ What happens when executor runs out of memory?

👉 Data spills to disk or task fails.

---

## ❓ Difference between execution and storage memory?

- Execution → Computation
- Storage → Cache

---

## ❓ What is unified memory?

👉 Shared pool for execution + storage.

---

# 🚀 Final Summary

```
Driver → Controls execution
Executor → Processes data

Execution Memory → Computation
Storage Memory → Cache

OOM → Happens when memory exceeded
Spill → Happens when memory insufficient
```

---

# 🔥 Golden Rules

- Never use collect() on big data  
- Broadcast small tables  
- Tune partitions  
- Monitor Spark UI  
- Avoid unnecessary caching  

# 🚀 Advanced Spark Engineering (Executor Sizing, Memory Internals, Debugging, Mock Interview)

---

# 1️⃣ Executor Sizing Strategy (Real Cluster Planning 🔥)

---

## 📌 Goal

Maximize:

- Parallelism  
- Resource utilization  
- Stability (avoid OOM)  

---

## 🔹 Key Factors

- Total cluster memory  
- Total CPU cores  
- Workload type (batch / streaming)  
- Data size  

---

## 🔹 Example Cluster

```
Cluster:
- 5 Nodes
- Each Node: 16 GB RAM, 8 Cores
```

---

## 🔹 Step 1: Reserve Memory for OS

```
16 GB → ~14 GB usable
```

---

## 🔹 Step 2: Decide Executor Memory

👉 Best Practice:

```
~4–6 GB per executor
```

---

## 🔹 Step 3: Decide Executor Cores

👉 Rule:

```
2–5 cores per executor
```

---

## 🔹 Example Configuration

```
Executors per node = 2
Executor memory = 6 GB
Executor cores = 3
```

---

## 🔹 Why Not Use All Cores?

- Too many cores → GC overhead  
- Too few cores → underutilization  

---

## 🔹 Formula (Interview Favorite 🔥)

```
Total Executors = Total Cores / Cores per Executor
```

---

## 🔹 Key Tips

- Avoid very large executors  
- Avoid very small executors  
- Balance memory + CPU  

---

# 2️⃣ Spark Memory Internals (Off-Heap + Tungsten)

---

# 🔹 On-Heap vs Off-Heap

---

## On-Heap Memory

- Managed by JVM  
- Subject to GC (Garbage Collection)  

---

## Off-Heap Memory

- Managed outside JVM  
- Used by Tungsten  

---

## 🔥 Why Off-Heap?

- Reduces GC overhead  
- Faster memory access  
- Better performance  

---

# 🔹 Tungsten Execution Engine

---

## 📌 What Tungsten Does

- Uses binary format (no Java objects)  
- Uses off-heap memory  
- Whole-stage code generation  

---

## 🔹 Benefits

- Less GC  
- Faster CPU usage  
- Efficient memory usage  

---

## 🔹 Example

Without Tungsten:

- Java objects → heavy memory  

With Tungsten:

- Compact binary format  

---

# 🔥 Whole-Stage Code Generation

Spark converts query into:

```
Optimized Java bytecode
```

Result:

- Faster execution  
- Less overhead  

---

# 3️⃣ Real-World Debugging Scenarios

---

# 🔥 Scenario 1: Driver OOM

---

## Problem

```python
df.collect()
```

---

## Symptom

- Driver crashes  
- OutOfMemoryError  

---

## Fix

```python
df.limit(100).collect()
```

OR

- Write to storage instead  

---

# 🔥 Scenario 2: Executor OOM (Shuffle)

---

## Problem

```python
df.groupBy("id").count()
```

---

## Symptom

- Spill to disk  
- Task failure  

---

## Fix

- Increase executor memory  
- Reduce partitions  
- Use aggregation optimization  

---

# 🔥 Scenario 3: Data Skew

---

## Problem

```
Key "A" → 90% data
```

---

## Symptom

- One task slow  
- Others idle  

---

## Fix

- Salting  
- Broadcast join  
- AQE  

---

# 🔥 Scenario 4: Too Many Small Files

---

## Problem

- 200 partitions → 200 files  

---

## Fix

```python
df.coalesce(10).write.parquet()
```

---

# 🔥 Scenario 5: Slow Join

---

## Problem

```python
df1.join(df2, "id")
```

---

## Fix

```python
df1.join(broadcast(df2), "id")
```

---

# 4️⃣ Mock Interview (Real Practice 🔥)

---

## ❓ Q1: How do you decide executor size?

👉 Based on cluster resources, workload, and avoiding GC overhead.

---

## ❓ Q2: Why does Spark use off-heap memory?

👉 To reduce GC and improve performance.

---

## ❓ Q3: Difference between driver and executor memory?

- Driver → Metadata + results  
- Executor → Data processing  

---

## ❓ Q4: How to debug slow job?

👉 Use Spark UI → Check stages, tasks, shuffle.

---

## ❓ Q5: Why is shuffle expensive?

👉 Disk + network + serialization.

---

## ❓ Q6: How to handle skew?

👉 Salting + AQE + broadcast join.

---

## ❓ Q7: What happens when executor fails?

👉 Spark recomputes using lineage.

---

## ❓ Q8: Why broadcast join is faster?

👉 Avoids shuffle.

---

## ❓ Q9: What is spill?

👉 Data moved from memory to disk.

---

## ❓ Q10: What is Tungsten?

👉 Execution engine for memory & CPU optimization.

---

# 🎯 Interview Strategy

---

## Always Answer Like This:

1. Explain concept  
2. Give example  
3. Explain problem  
4. Give optimization  

---

# 🚀 Final Summary

```
Executor Sizing → Balance CPU + Memory
Off-Heap → Reduce GC
Tungsten → Faster execution
Debugging → Spark UI + logs
Optimization → Reduce shuffle + handle skew
```

---

# 🔥 Golden Rules

- Avoid large executors  
- Avoid collect()  
- Use broadcast wisely  
- Monitor Spark UI  
- Tune partitions carefully  